In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from imblearn.over_sampling import SMOTE
import optuna
import matplotlib.pyplot as plt

In [2]:
# Load the dataset
df = pd.read_csv(r'C:\Users\khiew\Downloads\FYP Reduced Dataset.csv')

In [3]:
# Drop diseases with less than 500 instances
disease_counts = df['diseases'].value_counts()
valid_diseases = disease_counts[disease_counts >= 500].index
df = df[df['diseases'].isin(valid_diseases)]

# Assuming that the target variable is 'diseases' and all other variables are input features
X = df.drop('diseases', axis=1)
y = df['diseases']

# Encode the target variable (diseases) if it's a categorical variable
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=0.2, random_state=42)
num_classes = len(np.unique(y_train))
input_dim = X_train.shape[1]
print("Number of remaining classes in training set:", len(np.unique(y_train)))
print("Number of rows left:", len(df))

Number of remaining classes in training set: 201
Number of rows left: 168499


In [4]:
# Optuna objective function
def objective(trial):
    model = keras.Sequential()
    model.add(layers.Input(shape=(input_dim,)))

    # Suggest hyperparameters
    n_layers = trial.suggest_int('n_layers', 1, 3)
    for i in range(n_layers):
        num_units = trial.suggest_int(f'n_units_{i}', 64, 256, step=64)
        activation = trial.suggest_categorical(f'activation_{i}', ['relu', 'tanh'])
        model.add(layers.Dense(num_units, activation=activation))
        dropout_rate = trial.suggest_float(f'dropout_rate_{i}', 0.2, 0.5)
        model.add(layers.Dropout(dropout_rate))

    model.add(layers.Dense(num_classes, activation='softmax'))

    learning_rate = trial.suggest_float('learning_rate', 1e-4, 1e-2, log=True)
    optimizer = keras.optimizers.Adam(learning_rate=learning_rate)

    model.compile(optimizer=optimizer,
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])

    history = model.fit(X_train, y_train,
                        validation_split=0.2,
                        epochs=10,
                        batch_size=trial.suggest_categorical('batch_size', [32, 64, 128]),
                        verbose=0)

    val_accuracy = max(history.history['val_accuracy'])
    return val_accuracy

In [5]:
# Create Optuna study for optimization with persistent storage
study = optuna.create_study(
    direction='maximize',  # Assuming you are maximizing accuracy
    study_name="DNN_diseases_symptoms_dropextremelymore500withoutSMOTE_study", 
    storage=r"sqlite:///C:/Users/khiew/Downloads/dnn.db", 
    load_if_exists=True  # Load the study if it already exists, to resume from the last trial
)

# Optimize the study with your objective function, you can adjust the n_trials as needed
study.optimize(objective, n_trials=20)
# Print the best trial and hyperparameters
print("\nBest Trial:")
print(study.best_trial)
print("Best Hyperparameters:")
print(study.best_trial.params)

[I 2025-04-27 16:15:03,336] A new study created in RDB with name: DNN_diseases_symptoms_dropextremelymore500withoutSMOTE_study
[I 2025-04-27 16:15:40,095] Trial 0 finished with value: 0.40827152132987976 and parameters: {'n_layers': 1, 'n_units_0': 128, 'activation_0': 'tanh', 'dropout_rate_0': 0.3648440159189855, 'learning_rate': 0.00044902004833737626, 'batch_size': 32}. Best is trial 0 with value: 0.40827152132987976.
[I 2025-04-27 16:16:06,569] Trial 1 finished with value: 0.40163204073905945 and parameters: {'n_layers': 3, 'n_units_0': 128, 'activation_0': 'relu', 'dropout_rate_0': 0.33093982601169725, 'n_units_1': 192, 'activation_1': 'relu', 'dropout_rate_1': 0.4492552510370675, 'n_units_2': 128, 'activation_2': 'tanh', 'dropout_rate_2': 0.49812448804963405, 'learning_rate': 0.00025544711879873355, 'batch_size': 64}. Best is trial 0 with value: 0.40827152132987976.
[I 2025-04-27 16:16:51,585] Trial 2 finished with value: 0.40522995591163635 and parameters: {'n_layers': 2, 'n_uni


Best Trial:
FrozenTrial(number=13, state=TrialState.COMPLETE, values=[0.4094584584236145], datetime_start=datetime.datetime(2025, 4, 27, 16, 21, 21, 113706), datetime_complete=datetime.datetime(2025, 4, 27, 16, 21, 43, 921937), params={'n_layers': 1, 'n_units_0': 128, 'activation_0': 'tanh', 'dropout_rate_0': 0.4963570360630034, 'learning_rate': 0.00017867408440794082, 'batch_size': 64}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_layers': IntDistribution(high=3, log=False, low=1, step=1), 'n_units_0': IntDistribution(high=256, log=False, low=64, step=64), 'activation_0': CategoricalDistribution(choices=('relu', 'tanh')), 'dropout_rate_0': FloatDistribution(high=0.5, log=False, low=0.2, step=None), 'learning_rate': FloatDistribution(high=0.01, log=True, low=0.0001, step=None), 'batch_size': CategoricalDistribution(choices=(32, 64, 128))}, trial_id=216, value=None)
Best Hyperparameters:
{'n_layers': 1, 'n_units_0': 128, 'activation_0': 'tanh', 'dropout_rat